In [18]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

label_map = {'positive': 1, 'neutral': 0, 'negative': -1}

labels = []
tweets = []

with open('resources/tweets_train.txt', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        first_comma = line.index(',')
        label_str = line[:first_comma].strip().lower()
        tweet_text = line[first_comma+1:].strip()

        labels.append(label_map[label_str])
        tweets.append(tweet_text)

print(f"Total tweets: {len(tweets)}")
print(f"Sample label: {labels[0]} | Tweet: {tweets[0]}")
print(f"\nLabel distribution:\n{pd.Series(labels).value_counts()}")

preprocessed = [preprocess(tweet) for tweet in tweets]

print("\nOriginal  :", tweets[0])
print("Processed :", preprocessed[0])

vectorizer = CountVectorizer(max_features=500)
count_matrix = vectorizer.fit_transform(preprocessed)

print(f"\nWord count matrix shape: {count_matrix.shape}")

count_vectorized_df = pd.DataFrame.sparse.from_spmatrix(
    count_matrix,
    columns=vectorizer.get_feature_names_out()
)

print("\nDataFrame head:")
print(count_vectorized_df.head())

[nltk_data] Downloading package punkt to /home/oelhaouc/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/oelhaouc/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/oelhaouc/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Total tweets: 6588
Sample label: 1 | Tweet: Gas by my house hit $3.39!!!! I'm going to Chapel Hill on Sat. :)

Label distribution:
 0    3236
 1    2413
-1     939
Name: count, dtype: int64

Original  : Gas by my house hit $3.39!!!! I'm going to Chapel Hill on Sat. :)
Processed : ga hous hit 339 im go chapel hill sat

Word count matrix shape: (6588, 500)

DataFrame head:
   10  10th  12  15  15th  16  19th  1st  2012  22  ...  world  would  yall  \
0   0     0   0   0     0   0     0    0     0   0  ...      0      0     0   
1   0     0   0   0     0   0     0    0     0   0  ...      0      0     0   
2   0     0   0   0     0   0     0    0     0   0  ...      0      0     0   
3   0     0   0   0     0   0     0    0     0   0  ...      0      0     0   
4   0     0   0   0     0   0     0    0     0   0  ...      0      0     0   

   ye  year  yesterday  yet  york  young  your  
0   0     0          0    0     0      0     0  
1   0     0          0    0     0      0     0  
2   

In [19]:
fourth_tweet = count_vectorized_df.iloc[3]
fourth_nonzero = fourth_tweet[fourth_tweet > 0]

print("Original  :", tweets[3])
print("Processed :", preprocessed[3])
print("Token counts:\n", fourth_nonzero)

Original  : Iranian general says Israel's Iron Dome can't deal with their missiles (keep talking like that and we may end up finding out)
Processed : iranian gener say israel iron dome cant deal missil keep talk like may end find
Token counts:
 cant    1
deal    1
end     1
find    1
keep    1
like    1
may     1
say     1
talk    1
Name: 3, dtype: Sparse[int64, 0]


In [20]:
word_freq = count_vectorized_df.sum().sort_values(ascending=False)

print(word_freq.head(15))

tomorrow    1126
go           733
day          667
night        641
may          533
tonight      501
see          439
time         429
im           422
get          398
today        389
game         382
saturday     379
friday       375
sunday       368
dtype: Sparse[int64, 0]


In [21]:
count_vectorized_df['label'] = labels

print("\n--- Final DataFrame (first 5 rows, last 3 cols + label) ---")
print(count_vectorized_df.iloc[:5, -4:])

print("\nLabel counts:")
print(count_vectorized_df['label'].value_counts())


--- Final DataFrame (first 5 rows, last 3 cols + label) ---
   york  young  your  label
0     0      0     0      1
1     0      0     0     -1
2     0      0     0     -1
3     0      0     0     -1
4     0      0     0      1

Label counts:
label
 0    3236
 1    2413
-1     939
Name: count, dtype: int64
